# ERK-KTR Full FOV Stimulation Pipeline
This notebook showcases how to use the ERK-KTR full FOV stimulation pipeline. The pipeline is designed to simulate the full field of view (FOV) stimulation of a cells with the ERK-KTR biosensor. As it is a demo experiment, the pipeline runs on the demo hardware provided by MicroManager. 

### Import required libraries

In [1]:
import os
import time
from rtm_pymmcore.data_structures import Channel, StimTreatment
import rtm_pymmcore.utils as utils
from pprint import pprint
import pandas as pd

### Experimental Settings

In [2]:
# from rtm_pymmcore.microscope.MMDemo import MMDemo

# mic = MMDemo()
# mic.mmc.setChannelGroup("Channel")

# for Jungfrau Microscope enable here:
# from rtm_pymmcore.microscope.Jungfrau import Jungfrau

# mic = Jungfrau()
# mic.mmc.setChannelGroup("TTL_ERK")

from rtm_pymmcore.microscope.Jungfrau import Jungfrau

mic = Jungfrau()
mic.mmc.setChannelGroup("TTL_ERK")

In [ ]:
## Configuration options
FIRST_FRAME_STIMULATION = 10
# N_FRAMES = 340
N_FRAMES_PHASE_1 = 100
N_FRAMES_PHASE_2 = 150


SLEEP_BEFORE_EXPERIMENT_START_in_H = 0

TIME_BETWEEN_TIMESTEPS = 60  # time in seconds between frames
TIME_PER_FOV = 3.75  # time in seconds per fov

ADD_STIM_EXPOSURE_GROUP = (
    False  # add an entry showing the last stimulation exposure for each FOV
)
REGULAR_SPACING_BETWEEN_STIMULATIONS = (
    False  # if True, the stim_timesteps will be evenly spaced
)

## Storage path for the experiment
base_path = "E:\\Cedric"
experiment_name = "2025-11-13_Priming_FGFR_U0126"
path = os.path.join(base_path, experiment_name)


# Define Channels for which Images are taken. If no power is defined, the default power of the device will be used,
# for example, see the second channel "Cy5" below. The default power is set in the GUI
# Define Channels for which Images are taken. If no power is defined, the default power of the device will be used,
# for example, see the second channel "Cy5" below. The default power is set in the GUI
channels = []
channels.append(Channel(name="miRFP", exposure=300))
channels.append(Channel(name="mScarlet3", exposure=200))

# Channel to check for the expression of the optogenetic marker, can be used if it the marker is in the same channel as the stimulation channel.
channel_optocheck = Channel(name="mCitrine", exposure=600)
optocheck_timepoints = (149,)

condition = [
    "FGFR_PreStim",
    "FGFR_NoPreStim",
    "FGFR_U0126_PreStim",
    "FGFR_U0126_NoPreStim",
]
condition = [
    cond for cond in condition for _ in range(4)
] * 1  # means that we have 6 FOV per well, and 3 wells, so first 6 FOVs with FGFR,


# condition = ["optoFGFR_high"] * 24 + ["optoFGFR"] * 24 # Example of adding multiple conditions to the dataframe. n repreats the amount of times the condition is repeated.


n_fovs_per_well = None  ## change this variable to the amount of fovs that you have per well. Set to None if you are not working with wellplate.


# Stimulation parameters for optogenetics. The stimulation will be repeated for each condition.

stim_exposures_timesteps_phase_1 = [
    {
        "ramp_pattern_name": "Sustained",
        "stim_timestep": tuple(range(10, 100, 1)),
        "stim_exposure_list": (100,) * 90,
    }
]

stim_exposures_timesteps_phase_2 = [
    {
        "ramp_pattern_name": "Sustained",
        "stim_timestep": tuple(range(30, 120, 1)),
        "stim_exposure_list": (100,) * 90,
    }
]


for stim_exposures_timesteps in [
    stim_exposures_timesteps_phase_1,
    stim_exposures_timesteps_phase_2,
]:
    utils.fix_tuples_in_stim_exposure_list(stim_exposures_timesteps)
    utils.add_stim_parameters_to_stim_exposures_timesteps(stim_exposures_timesteps)
    utils.print_stim_exposures_timesteps(stim_exposures_timesteps)

## Define the Tools that you are using for the experiment
from rtm_pymmcore.stimulation.base_stimulation import StimWholeFOV
from rtm_pymmcore.tracking.trackpy import TrackerTrackpy
from rtm_pymmcore.feature_extraction.erk_ktr import FE_ErkKtr
from rtm_pymmcore.feature_extraction.optocheck_fe import OptoCheckFE
from rtm_pymmcore.segmentation.cellpose_v4 import CellposeV4

segmentators = [
    {
        "name": "labels",
        "class": CellposeV4(
            custom_model_path="E:\\models\\cellpose\\LifeActH2B_mixed_with_only_H2B_v1",
            min_size=100,
        ),
        "use_channel": 0,
        "save_tracked": True,
    },
]

stimulator = StimWholeFOV()
feature_extractor = FE_ErkKtr("labels")
tracker = TrackerTrackpy(search_range=35)
optocheck = OptoCheckFE(used_mask="labels")


from rtm_pymmcore.img_processing_pip import ImageProcessingPipeline

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=feature_extractor,
    tracker=tracker,
    stimulator=stimulator,
    feature_extractor_optocheck=optocheck,
)
mic.set_pipeline(pipeline=pipeline)

Pattern Name:  Sustained
100 at 10
100 at 11
100 at 12
100 at 13
100 at 14
100 at 15
100 at 16
100 at 17
100 at 18
100 at 19
100 at 20
100 at 21
100 at 22
100 at 23
100 at 24
100 at 25
100 at 26
100 at 27
100 at 28
100 at 29
100 at 30
100 at 31
100 at 32
100 at 33
100 at 34
100 at 35
100 at 36
100 at 37
100 at 38
100 at 39
100 at 40
100 at 41
100 at 42
100 at 43
100 at 44
100 at 45
100 at 46
100 at 47
100 at 48
100 at 49
100 at 50
100 at 51
100 at 52
100 at 53
100 at 54
100 at 55
100 at 56
100 at 57
100 at 58
100 at 59
100 at 60
100 at 61
100 at 62
100 at 63
100 at 64
100 at 65
100 at 66
100 at 67
100 at 68
100 at 69
100 at 70
100 at 71
100 at 72
100 at 73
100 at 74
100 at 75
100 at 76
100 at 77
100 at 78
100 at 79
100 at 80
100 at 81
100 at 82
100 at 83
100 at 84
100 at 85
100 at 86
100 at 87
100 at 88
100 at 89
100 at 90
100 at 91
100 at 92
100 at 93
100 at 94
100 at 95
100 at 96
100 at 97
100 at 98
100 at 99

Pattern Name:  Sustained
100 at 30
100 at 31
100 at 32
100 at 33
100 at 34

### GUI - Napari Micromanager

#### Load GUI

In [4]:
### Base GUI ###
from napari_micromanager import MainWindow
import napari

viewer = napari.Viewer()
mm_wdg = MainWindow(viewer)
mm_wdg._mmc = mic.mmc
viewer.window.add_dock_widget(mm_wdg)
data_mda_fovs = None
load_from_file = False

Please execute the following code block, if you would like to set a custom ROI (e.g. smaller illumination than the full field of view of the camera). Execute it after you have started the camera once in the GUI. 

In [ ]:
if mic.SET_ROI_REQUIRED:
    mic.mmc.clearROI()
    mic.mmc.setROI(mic.ROI_X, mic.ROI_Y, mic.ROI_WIDTH, mic.ROI_HEIGHT)

### Map Experiment to FOVs

### Use FOVs to generate dataframe for acquisition

In [5]:
fovs = utils.generate_fov_objects(mic, viewer=viewer)
fovs

df_acquire = utils.generate_df_acquire(
    fovs,
    n_frames=N_FRAMES_PHASE_1,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    time_per_fov=TIME_PER_FOV,
    channels=channels,
    phase_name="PreDrug",
    phase_id=0,
    condition=condition,
)
df_acquire = utils.apply_stim_treatments_to_df_acquire(
    df_acquire,
    stim_exposures_timesteps_phase_1,
    condition,
    n_fovs_per_well=n_fovs_per_well,
    add_stim_exposure_group=ADD_STIM_EXPOSURE_GROUP,
    regular_spacing_between_stimulations=REGULAR_SPACING_BETWEEN_STIMULATIONS,
)
df_acquire

c:\Users\Jungfrau\Documents\alandolt\code\rtm-pymmcore\rtm_pymmcore\utils.py:106: FutureWarning: The `_dock_widgets` property is private and should not be used in any plugin code. Please use the `dock_widgets` property instead.
  data_mda_fovs = viewer.window._dock_widgets["MDA"].widget().value().stage_positions


Total Experiment Time: 1.65h
Doing 4 experiment per stim condition


,fov_object,fov,fov_x,fov_y,fov_name,timestep,time,channels,fname,cell_line,...,ramp_pattern_name,stim_timestep,stim_exposure_list,stim_power,stim_channel_name,stim_channel_group,stim_channel_device_name,stim_channel_power_property_name,stim_exposure,stim
0,<rtm_pymmcore.data_structures.Fov object at 0x...,0,44157.9,-11508.6,0,0,0.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",000_00_00000,FGFR_PreStim,...,Sustained,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
1,<rtm_pymmcore.data_structures.Fov object at 0x...,1,43949.1,-12853.7,1,0,0.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_00_00000,FGFR_PreStim,...,Sustained,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
2,<rtm_pymmcore.data_structures.Fov object at 0x...,2,42856.2,-12053.7,2,0,0.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",002_00_00000,FGFR_PreStim,...,Sustained,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
3,<rtm_pymmcore.data_structures.Fov object at 0x...,3,44285.8,-13540.2,3,0,0.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",003_00_00000,FGFR_PreStim,...,Sustained,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
4,<rtm_pymmcore.data_structures.Fov object at 0x...,4,44547.0,-14595.5,4,0,0.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",004_00_00000,FGFR_NoPreStim,...,Sustained,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1595,<rtm_pymmcore.data_structures.Fov object at 0x...,11,42917.6,-5434.4,11,99,5940.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",011_00_00099,FGFR_U0126_PreStim,...,Sustained,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,100.0,True
1596,<rtm_pymmcore.data_structures.Fov object at 0x...,12,43634.5,-6479.7,12,99,5940.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",012_00_00099,FGFR_U0126_NoPreStim,...,Sustained,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,100.0,True
1597,<rtm_pymmcore.data_structures.Fov object at 0x...,13,45495.8,-5080.3,13,99,5940.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",013_00_00099,FGFR_U0126_NoPreStim,...,Sustained,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,100.0,True
1598,<rtm_pymmcore.data_structures.Fov object at 0x...,14,45177.0,-4070.4,14,99,5940.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",014_00_00099,FGFR_U0126_NoPreStim,...,Sustained,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,100.0,True


In [6]:
df_acquire.loc[(df_acquire["fov"].between(4, 7)), "stim_exposure"] = 0

df_acquire.loc[(df_acquire["fov"].between(12, 15)), "stim_exposure"] = 0

df_acquire.loc[(df_acquire["fov"].between(4, 7)), "stim_power"] = 0

df_acquire.loc[(df_acquire["fov"].between(12, 15)), "stim_power"] = 0

df_acquire

,fov_object,fov,fov_x,fov_y,fov_name,timestep,time,channels,fname,cell_line,...,ramp_pattern_name,stim_timestep,stim_exposure_list,stim_power,stim_channel_name,stim_channel_group,stim_channel_device_name,stim_channel_power_property_name,stim_exposure,stim
0,<rtm_pymmcore.data_structures.Fov object at 0x...,0,44157.9,-11508.6,0,0,0.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",000_00_00000,FGFR_PreStim,...,Sustained,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
1,<rtm_pymmcore.data_structures.Fov object at 0x...,1,43949.1,-12853.7,1,0,0.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_00_00000,FGFR_PreStim,...,Sustained,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
2,<rtm_pymmcore.data_structures.Fov object at 0x...,2,42856.2,-12053.7,2,0,0.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",002_00_00000,FGFR_PreStim,...,Sustained,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
3,<rtm_pymmcore.data_structures.Fov object at 0x...,3,44285.8,-13540.2,3,0,0.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",003_00_00000,FGFR_PreStim,...,Sustained,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
4,<rtm_pymmcore.data_structures.Fov object at 0x...,4,44547.0,-14595.5,4,0,0.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",004_00_00000,FGFR_NoPreStim,...,Sustained,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",0,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1595,<rtm_pymmcore.data_structures.Fov object at 0x...,11,42917.6,-5434.4,11,99,5940.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",011_00_00099,FGFR_U0126_PreStim,...,Sustained,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,100.0,True
1596,<rtm_pymmcore.data_structures.Fov object at 0x...,12,43634.5,-6479.7,12,99,5940.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",012_00_00099,FGFR_U0126_NoPreStim,...,Sustained,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",0,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,True
1597,<rtm_pymmcore.data_structures.Fov object at 0x...,13,45495.8,-5080.3,13,99,5940.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",013_00_00099,FGFR_U0126_NoPreStim,...,Sustained,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",0,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,True
1598,<rtm_pymmcore.data_structures.Fov object at 0x...,14,45177.0,-4070.4,14,99,5940.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",014_00_00099,FGFR_U0126_NoPreStim,...,Sustained,"(10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 2...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",0,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,True


In [7]:
df_acquire_2 = utils.generate_df_acquire(
    fovs,
    n_frames=N_FRAMES_PHASE_2,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    time_per_fov=TIME_PER_FOV,
    channels=channels,
    channel_optocheck=channel_optocheck,
    optocheck_timepoints=optocheck_timepoints,
    phase_name="PostDrug",
    phase_id=1,
    condition=condition,
)
df_acquire_2 = utils.apply_stim_treatments_to_df_acquire(
    df_acquire_2,
    stim_exposures_timesteps_phase_2,
    condition,
    n_fovs_per_well=n_fovs_per_well,
    add_stim_exposure_group=ADD_STIM_EXPOSURE_GROUP,
    regular_spacing_between_stimulations=REGULAR_SPACING_BETWEEN_STIMULATIONS,
)
df_acquire_2

Total Experiment Time: 2.4833333333333334h
Doing 4 experiment per stim condition


,fov_object,fov,fov_x,fov_y,fov_name,timestep,time,channels,fname,cell_line,...,ramp_pattern_name,stim_timestep,stim_exposure_list,stim_power,stim_channel_name,stim_channel_group,stim_channel_device_name,stim_channel_power_property_name,stim_exposure,stim
0,<rtm_pymmcore.data_structures.Fov object at 0x...,0,44157.9,-11508.6,0,0,0.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",000_01_00000,FGFR_PreStim,...,Sustained,"(30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 4...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
1,<rtm_pymmcore.data_structures.Fov object at 0x...,1,43949.1,-12853.7,1,0,0.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_01_00000,FGFR_PreStim,...,Sustained,"(30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 4...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
2,<rtm_pymmcore.data_structures.Fov object at 0x...,2,42856.2,-12053.7,2,0,0.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",002_01_00000,FGFR_PreStim,...,Sustained,"(30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 4...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
3,<rtm_pymmcore.data_structures.Fov object at 0x...,3,44285.8,-13540.2,3,0,0.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",003_01_00000,FGFR_PreStim,...,Sustained,"(30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 4...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
4,<rtm_pymmcore.data_structures.Fov object at 0x...,4,44547.0,-14595.5,4,0,0.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",004_01_00000,FGFR_NoPreStim,...,Sustained,"(30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 4...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2395,<rtm_pymmcore.data_structures.Fov object at 0x...,11,42917.6,-5434.4,11,149,8940.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",011_01_00149,FGFR_U0126_PreStim,...,Sustained,"(30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 4...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
2396,<rtm_pymmcore.data_structures.Fov object at 0x...,12,43634.5,-6479.7,12,149,8940.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",012_01_00149,FGFR_U0126_NoPreStim,...,Sustained,"(30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 4...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
2397,<rtm_pymmcore.data_structures.Fov object at 0x...,13,45495.8,-5080.3,13,149,8940.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",013_01_00149,FGFR_U0126_NoPreStim,...,Sustained,"(30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 4...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
2398,<rtm_pymmcore.data_structures.Fov object at 0x...,14,45177.0,-4070.4,14,149,8940.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",014_01_00149,FGFR_U0126_NoPreStim,...,Sustained,"(30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 4...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False


### Run experiment

In [8]:
for _ in range(0, SLEEP_BEFORE_EXPERIMENT_START_in_H * 3600):
    time.sleep(1)

try:
    mm_wdg._core_link.cleanup()

except:
    pass

mic.run_experiment(df_acquire)

In [9]:
mic.run_experiment(df_acquire_2)

In [10]:
mic.post_experiment()
time.sleep(10)

utils.generate_exp_data_from_tracks(path)

from napari_micromanager._core_link import CoreViewerLink

if "viewer" in locals():
    mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

### Function to re-connect link with GUI if manually broken

The following functions can be used to manually re-make the connection between the GUI and the running rtm-pymmcore script. However, normally you don't need to execute them. 

In [18]:
### Manually reconnect pymmcore with napari-micromanager
from napari_micromanager._core_link import CoreViewerLink

mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

The following code block can be used to manually break the connection between GUI and Jupyter Notebook:


In [ ]:
### Break connection
# mm_wdg._core_link.cleanup()